# 1유형

In [ ]:
import pandas as pd
# 1. 주어진 trash bag 데이터 세트는 지역별 종량제 봉투 가격을 나타낸다. 
# 가격 컬럼은 각 행의 조건을 만족하는 해당 용량의 종량제 봉투가 존재하면 가격을 값으로, 존재하지 않으면 0을 값으로 갖는다.
# 이때 용도가 '음식물쓰레기'이고 사용 대상이 '가정용'인 2L 봉투 가격의 평균을 소수점을 버린 후 정수로 출력하시오.
df = pd.read_csv('datasets/Part5/501_trash_bag.csv', encoding='euc-kr')
result = df.loc[(df['용도']=='음식물쓰레기')&(df['사용대상']=='가정용')&(df['2L가격']!=0)]['2L가격'].mean()
print(int(result))

In [ ]:
# 2. BMI지수는 몸무게(kg)를 키(m)의 제곱으로 나누어 구하며, BMI 값에 따른 비만도 분류는 다음과 같다.
# | BMI지수 범위         | 비만도 분류    |
# |---------------------|---------------|
# | 18.5 미만           | 저체중         |
# | 18.5 이상 ~ 23 미만 | 정상           |
# | 23 이상 ~ 25 미만   | 과체중         |
# | 25 이상 ~ 30 미만   | 경도비만       |
# | 30 이상             | 중등도비만     |
# 이때 주어진 bmi 데이터 세트에서 비만도가 정상에 속하는 인원 수와 과체중에 속하는 인원 수의 차이를 정수로 출력하시오.
df = pd.read_csv('datasets/Part5/502_bmi.csv')
df['bmi'] = df['Weight'] / (df['Height']/100)**2
normal = len(df.loc[(df['bmi'] >= 18.5)&(df['bmi'] < 23)])
over = len(df.loc[(df['bmi'] >= 23)&(df['bmi'] < 25)])
answer = normal - over
print(answer)

In [ ]:
# 3. 주어진 students 데이터 세트는 각 학교의 학년별 총 전입학생, 총 전출학생, 전체 학생 수를 나타낸다.
# 순 전입학생 수는 총 전입학생 수에서 총 전출학생 수를 빼서 구할 수 있다.
# 순 전입학생이 가장 많은 학교의 전체 학생 수를 구하시오.
df = pd.read_csv('datasets/Part5/503_students.csv', encoding='euc-kr')
df['순 전입학생'] = df['총 전입학생'] - df['총 전출학생']
df.groupby('학교')['순 전입학생'].sum().idxmax()
result = df.loc[df['학교']=='A']['전체 학생 수'].sum()
# df.groupby('학교')[['순 전입학생','전체 학생 수']].sum().sort_values(by='순 전입학생').iloc[-1,1]
print(result)

# 2유형

In [ ]:
# 1. 다음은 Used car 데이터 세트이다. 주어진 훈련 데이터 세트를 활용하여 중고차의 판매 가격을 예측하고 해당 예측 결과를 csv 파일로 제출하시오.
# 결과 제출 양식: 제출한 예측값의 rmse 평가지표 값을 통해 영역별 배점에 따라 최종 점수가 반영될 예정
# | id | price |
# |----|-------|
# | 1  | 12500 |
# | 2  | 16500 |
# | 3  | 11000 |
# | ... | ... |
# [결과 제출 양식]

# | 변수           | 설명          |
# |---------------|---------------|
# | id            | 중고차 ID 번호 |
# | model         | 차량 모델명    |
# | year          | 차량 등록 연도 |
# | transmission  | 변속기 종류    |
# | mileage       | 주행 거리      |
# | fuelType      | 엔진 연료 종류 |
# | tax           | 도로세        |
# | mpg           | 갤런당 마일(miles per gallon), 연비 |
# | engineSize    | 엔진 크기     |
# | price         | 가격(파운드)  |
# [Used Car 데이터 세트 변수 설명]

In [ ]:
import pandas as pd
x_train = pd.read_csv('datasets/Part5/504_x_train.csv')
y_train = pd.read_csv('datasets/Part5/504_y_train.csv')
x_test = pd.read_csv('datasets/Part5/504_x_test.csv')

# x_train.info(), y_train.info(), x_test.info()
# x_train.shape, y_train.shape, x_test.shape
# x_train.isnull().sum(), y_train.isnull().sum(), x_test.isnull().sum()

x_train = x_train.drop(columns=['id'])
y = y_train['price']
x_test_id = x_test.pop('id')

from sklearn.preprocessing import MinMaxScaler, LabelEncoder
num_col = x_train.select_dtypes(exclude='object').columns
# num_col = num_col.drop('year')

Mm = MinMaxScaler()
x_train[num_col] = Mm.fit_transform(x_train[num_col])
x_test[num_col] = Mm.transform(x_test[num_col])

encoder = LabelEncoder()
for col in x_train.columns:
    if x_train[col].dtype == 'object':
        encoder.fit(pd.concat([x_train[col], x_test[col]]))
        x_train[col] = encoder.transform(x_train[col])
        x_test[col] = encoder.transform(x_test[col])

from sklearn.model_selection import train_test_split
x_train, x_val, y_train, y_val = train_test_split(x_train, y, test_size=0.2)

from sklearn.ensemble import RandomForestRegressor
model = RandomForestRegressor()
model.fit(x_train, y_train)
y_val_pred = model.predict(x_val)

from sklearn.metrics import root_mean_squared_error, r2_score
rmse = root_mean_squared_error(y_val, y_val_pred)
r2 = r2_score(y_val, y_val_pred)
print(rmse, r2)

pred = model.predict(x_test)
result = pd.DataFrame({'id':x_test_id, 'price':pred})
result.to_csv('result.csv', index=False)
# pd.read_csv('result.csv')

In [ ]:
import pandas as pd
x_train = pd.read_csv('datasets/Part5/504_x_train.csv')
y_train = pd.read_csv('datasets/Part5/504_y_train.csv')
x_test = pd.read_csv('datasets/Part5/504_x_test.csv')

COL_DEL = ['id']
COL_NUM = ['year','mileage','tax','mpg','engineSize']
COL_CAT = ['model', 'transmission', 'fuelType']
COL_Y = ['price']

from sklearn.model_selection import train_test_split
x_tr, x_val, y_tr, y_val = train_test_split(x_train[COL_NUM+COL_CAT], y_train[COL_Y].values.ravel(), test_size=0.3)

from sklearn.preprocessing import StandardScaler, LabelEncoder
scaler = StandardScaler()
scaler.fit(x_tr[COL_NUM])

x_tr[COL_NUM] = scaler.transform(x_tr[COL_NUM])
x_val[COL_NUM] = scaler.transform(x_val[COL_NUM])
x_test[COL_NUM] = scaler.transform(x_test[COL_NUM])

X = pd.concat([x_train[COL_CAT], x_test[COL_CAT]])
for col in COL_CAT:
    le = LabelEncoder()
    le.fit(X[col])
    x_tr[col] = le.transform(x_tr[col])
    x_val[col] = le.transform(x_val[col])
    x_test[col] = le.transform(x_test[col])

from sklearn.ensemble import RandomForestRegressor
modelRF = RandomForestRegressor(random_state=123)
modelRF.fit(x_tr, y_tr)

from xgboost import XGBRegressor
modelXGB = XGBRegressor(objective='reg:squarederror', random_state=123)
modelXGB.fit(x_tr, y_tr)

y_val_predRF = modelRF.predict(x_val)
y_val_predXGB = modelXGB.predict(x_val)

from sklearn.metrics import root_mean_squared_error, r2_score
rmseRF = root_mean_squared_error(y_val, y_val_predRF)
r2RF = r2_score(y_val, y_val_predRF)
# print(rmseRF, r2RF)

rmseXGB = root_mean_squared_error(y_val, y_val_predXGB)
r2XGB = r2_score(y_val, y_val_predXGB)
# print(rmseXGB, r2XGB)

pred = modelRF.predict(x_test[COL_NUM+COL_CAT])
result = pd.DataFrame({'id':x_test['id'], 'price':pred})
result.to_csv('result.csv', index=False)